<a href="https://colab.research.google.com/github/mdonbruce/AspNetDocs/blob/master/04_instructor_executed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: Custom Exceptions in Business Logic — Instructor (Executed)

**Objective:** Use custom exceptions to classify loan application rejections.

**Dataset:** `loan_applications_large.csv`

This notebook is a complete reference solution with outputs.

In [ ]:
import pandas as pd, numpy as np
from collections import Counter

df = pd.read_csv('loan_applications_large.csv')
df.head()

,application_id,timestamp,annual_income,credit_score,requested_amount,debt_to_income,employment_type,state
0,APP-000001,2025-07-23T13:32:21,49766.91,636,11580.0,0.524,full_time,FL
1,APP-000002,2025-06-04T06:16:18,59207.26,764,35970.0,0.201,full_time,WA
2,APP-000003,2025-04-11T11:36:40,61886.65,778,21130.0,0.373,unemployed,GA
3,APP-000004,2025-04-02T07:00:49,60252.05,765,38700.0,0.041,full_time,NC
4,APP-000005,2025-05-13T10:19:54,75294.45,676,44490.0,0.329,self_employed,FL


In [ ]:
class MissingDataError(Exception):
    def __init__(self, reason):
        super().__init__(reason)
        self.reason = reason

class EligibilityError(Exception):
    def __init__(self, reason):
        super().__init__(reason)
        self.reason = reason

def to_float(x):
    if x is None:
        return None
    s = str(x).strip().replace('$','').replace(',','')
    if s == '':
        return None
    try:
        v = float(s)
        if not np.isfinite(v):
            return None
        return float(v)
    except Exception:
        return None

def to_int(x):
    try:
        return int(x)
    except Exception:
        return None

def check_eligibility(row):
    inc = to_float(row['annual_income'])
    cs = to_int(row['credit_score'])
    dti = to_float(row['debt_to_income'])
    req = to_float(row['requested_amount'])
    if inc is None or cs is None or dti is None or req is None:
        raise MissingDataError('missing_or_malformed_fields')
    if inc < 25000:
        raise EligibilityError('income_below_min')
    if cs < 580 or cs > 850:
        raise EligibilityError('credit_out_of_range')
    if dti < 0 or dti > 0.45:
        raise EligibilityError('dti_too_high')
    if req <= 0 or req > 50000:
        raise EligibilityError('requested_amount_too_high')
    return True

In [ ]:
reasons = Counter()
approved = 0
for _, row in df.iterrows():
    try:
        if check_eligibility(row):
            approved += 1
    except (MissingDataError, EligibilityError) as e:
        reasons[e.reason] += 1
    except Exception:
        reasons['unexpected_error'] += 1

approved, reasons.most_common(10)

(3284,
 [('requested_amount_too_high', 1411),
  ('dti_too_high', 520),
  ('credit_out_of_range', 320),
  ('missing_or_malformed_fields', 259),
  ('income_below_min', 206)])